In [2]:
import pandas as pd

ex_path = r'/Users/eashanchawla/Documents/Field/grainger/grainger-esci-verification/data/raw/shopping_queries_dataset_examples.parquet'
pr_path = r'/Users/eashanchawla/Documents/Field/grainger/grainger-esci-verification/data/raw/shopping_queries_dataset_products.parquet'

df_examples = pd.read_parquet(ex_path)
df_products = pd.read_parquet(pr_path)

In [3]:
df_examples.head(10)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train
5,5,revent 80 cfm,0,B076Q7V5WX,us,E,0,1,train
6,6,revent 80 cfm,0,B075ZBF9HG,us,E,0,1,train
7,7,revent 80 cfm,0,B06W2LB17J,us,E,0,1,train
8,8,revent 80 cfm,0,B07JY1PQNT,us,E,0,1,train
9,9,revent 80 cfm,0,B01MZIK0PI,us,E,0,1,train


In [4]:
df_products.head(10)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,B079VKKJN7,"11 Degrees de los Hombres Playera con Logo, Ne...",Esta playera con el logo de la marca Carrier d...,11 Degrees Negro Playera con logo\nA estrenar ...,11 Degrees,Negro,es
1,B079Y9VRKS,Camiseta Eleven Degrees Core TS White (M),NaN,NaN,11 Degrees,Blanco,es
2,B07DP4LM9H,11 Degrees de los Hombres Core Pull Over Hoodi...,La sudadera con capucha Core Pull Over de 11 G...,11 Degrees Azul Core Pull Over Hoodie\nA estre...,11 Degrees,Azul,es
3,B07G37B9HP,11 Degrees Poli Panel Track Pant XL Black,NaN,NaN,11 Degrees,NaN,es
4,B07LCTGDHY,11 Degrees Gorra Trucker Negro OSFA (Talla úni...,NaN,NaN,11 Degrees,Negro (,es
5,B07MSD1JH3,"11 Degrees de los Hombres Optum Poly Joggers, ...",Los Optum Poly Joggers de 11 grados vienen con...,11 Degrees Negro Optum Poly Joggers\nA estrena...,11 Degrees,Negro,es
6,B07QKLGMHM,11 Degrees Core Zip Poly Top S Black,El Chándal ha sido diseñado con mangas largas ...,NaN,11 Degrees,Negro,es
7,B07S1VM815,11 Degrees Camiseta De Núcleo M Hot Red,NaN,NaN,11 Degrees,NaN,es
8,B07T1HCDXG,11 Degrees Trucker Cap - Black & White,NaN,NaN,11 Degrees,Black & White,es
9,B07VCV1LSQ,11 Degrees Chaqueta Espacial S Black,La chaqueta Space Puffer de 11 Degrees viene c...,11 Degrees Negro Chaqueta acolchada\nA estrena...,11 Degrees,Negro,es


In [5]:
df_merged = pd.merge(
    df_examples, df_products, how='left', on=['product_locale', 'product_id']
)

In [6]:
df_products.shape
df_examples.shape

(2621288, 9)

In [7]:
df_merged.shape

(2621288, 14)

1. Different product locales? 
2. Are there multiple descriptions for the same product id (same locale)? 
3. Can the same product id appear in different locales and languages? 
4. Look through random examples 
5. Does data need any cleaning? Seeing some weird foramtting examples: 2621283, 2621284 query_id

In [17]:
# df_merged['query_lower'] = df_merged['query'].str.lower()
# df_merged[(df_merged['query_lower'].str.contains('aa batteries', na=False)) & (df_merged['query_lower'].str.contains('100', na=False))]

In [20]:
df_subset = df_merged[df_merged['query'].isin(target_queries)]

In [26]:
df_subset.to_csv('../data/processed/subset_to_review.csv', index=False)

#### Are there multiple descriptions for the same product id same locale

So product locale and product id is safe to merge on. Left join won't explore

In [10]:
# are there any duplicate rows? 
df_products.duplicated().sum() # No duplicates

np.int64(0)

In [11]:
# is product locale and id safe to use as a composite key?
df_products.duplicated(subset=['product_locale', 'product_id'], keep=False).sum()

np.int64(0)

#### Can the same product appear in different locales / languages? 

In [13]:
locale_counts = df_products.groupby('product_id')['product_locale'].nunique()
locale_counts[locale_counts>1]

product_id
0007477155    2
0007977719    2
006029809X    2
0061992275    2
0062464310    2
             ..
B098JPS73M    2
B0992CF7XJ    2
B099MY9WF7    2
B099NB5CGK    2
B09B99LRD4    2
Name: product_locale, Length: 11567, dtype: int64

In [16]:
df_products[df_products['product_id'] == locale_counts[locale_counts > 1].index[16]]

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
605473,0307264874,We Tell Ourselves Stories in Order to Live: Co...,NaN,NaN,Everyman's Library,Tan,us
1190397,0307264874,We Tell Ourselves Stories in Order to Live: Co...,NaN,NaN,Everyman's Library,Tan,es


In [18]:
target_queries = [
    'aa batteries 100 pack', 'kodak photo paper 8.5 x 11 glossy', 'dewalt 8v max cordless screwdriver kit, gyroscopic'
]

df_target = df_merged[(df_merged['query'].isin(target_queries)) & (df_merged['esci_label'] == 'E')].copy()

In [19]:
df_target.shape

(24, 14)

In [20]:
df_target.loc[:, ['product_title', 'product_bullet_point', 'product_description']].isna().sum()

product_title            0
product_bullet_point     2
product_description     18
dtype: int64

In [26]:
for idx, row in df_target.sample(min(5, len(df_target))).iterrows():
    print(f'Query: {row['query']}')
    print(f'Title: {row['product_title']}')
    print(f'Description: {row['product_description']}')
    print(f'Bullets: {row['product_bullet_point']}')
    print('*' * 20)
    

Query: dewalt 8v max cordless screwdriver kit, gyroscopic
Title: DEWALT 8V MAX Cordless Screwdriver Kit, Gyroscopic, 1 Battery, Electric (DCF682N1)
Description: nan
Bullets: The cordless screwdriver features motion activation variable speed and reversing control for precise fastening control
Motion activated variable speed 0-430 rpm of the rechargeable screwdriver is made for fastening into wood, plastic, and light-gauge metal
The powered screwdriver allows illumination in confined areas without shadowing
Battery charge status on tool notifies when to charge packs
1/4-inch hex allows for quick screwdriver bit change and holds 1-inch bit tips
********************
Query: kodak photo paper 8.5 x 11 glossy
Title: KODAK Photo Paper Gloss 8.5"x11", 25 count, 48lb-180g/m2 weight, 6.5 mil thickness (41161 - 1912369),White
Description: nan
Bullets: Basic gloss paper for arts, craft and snapshots
Instant dry: No smearing or smudging
Guaranteed to work with any inkjet printer
When a picture on a 

In [23]:
df_products.head(5)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,B079VKKJN7,"11 Degrees de los Hombres Playera con Logo, Ne...",Esta playera con el logo de la marca Carrier d...,11 Degrees Negro Playera con logo\nA estrenar ...,11 Degrees,Negro,es
1,B079Y9VRKS,Camiseta Eleven Degrees Core TS White (M),NaN,NaN,11 Degrees,Blanco,es
2,B07DP4LM9H,11 Degrees de los Hombres Core Pull Over Hoodi...,La sudadera con capucha Core Pull Over de 11 G...,11 Degrees Azul Core Pull Over Hoodie\nA estre...,11 Degrees,Azul,es
3,B07G37B9HP,11 Degrees Poli Panel Track Pant XL Black,NaN,NaN,11 Degrees,NaN,es
4,B07LCTGDHY,11 Degrees Gorra Trucker Negro OSFA (Talla úni...,NaN,NaN,11 Degrees,Negro (,es


**To make the LLM reason, we will formulate the query in the following format:**

1. Extract information from the query -- Brand, Size, Quantity, Features
2. Extract infromation from the product title, description, bullet points, brand 
3. If query requires an attribute, and hte product text does not have that at all, assume that it matches 
4. If product includes extra items or features not mentioned in the query, this is acceptable. Do not flag as conflict
5. Only flag a conflict if product text explicitly contradicts Query requirement (query asks 100 pack, product is 50, or query asks Kodak product is dewalt)
6. Reformulate the query so it matches the product -- no extra words, follow same format, just fix the contradictory attribute 